In [16]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ds1 = pd.read_csv("DS1_all.csv")
ds2 = pd.read_csv("DS2_inspection.csv")

ds1["completion_date"] = pd.to_datetime(ds1["NSEARCH3_E"], errors="coerce")
ds1["age"] = (pd.Timestamp.today() - ds1["completion_date"]).dt.days / 365.25

ds2_enriched = ds2.merge(
    ds1[["OBJECTID", "SEARCH1_E", "SEARCH2_E", "NSEARCH4_E", "NSEARCH5_E", "age"]],
    on="OBJECTID", how="left"
)

In [17]:
import plotly.io as pio
import plotly.graph_objects as go

# ── App colour palette ────────────────────────────────────────────────────────
BG       = "#151617"          # presentation background
BG_PLOT  = "#0c111c"          # slightly lighter for plot area (app --bg-2)
TEXT0    = "#e6ecf5"
TEXT1    = "#aab4c4"
TEXT2    = "#6c7689"
ACCENT   = "#6ec1ff"          # app --accent  (replaces steelblue)
CRITICAL = "#E84545"          # app --critical (replaces tomato/red)
HIGH     = "#F37735"          # app --high
MODERATE = "#F5B642"          # app --moderate
LOW      = "#3FB6B0"          # app --low
BORDER   = "rgba(120,160,220,0.13)"
GRIDLINE = "rgba(140,200,255,0.07)"

_t = go.layout.Template()
_t.layout = go.Layout(
    paper_bgcolor=BG,
    plot_bgcolor=BG_PLOT,
    font=dict(color=TEXT0, family="ui-monospace, 'JetBrains Mono', 'SF Mono', Menlo, Consolas, monospace"),
    title=dict(font=dict(color=TEXT0, size=15)),
    xaxis=dict(gridcolor=GRIDLINE, linecolor=BORDER, zerolinecolor=GRIDLINE,
               tickfont=dict(color=TEXT1), title=dict(font=dict(color=TEXT1))),
    yaxis=dict(gridcolor=GRIDLINE, linecolor=BORDER, zerolinecolor=GRIDLINE,
               tickfont=dict(color=TEXT1), title=dict(font=dict(color=TEXT1))),
    legend=dict(bgcolor="rgba(21,22,23,0.85)", bordercolor=BORDER,
                font=dict(color=TEXT1)),
    colorway=[ACCENT, CRITICAL, HIGH, MODERATE, LOW, TEXT2],
    width=1200, height=800,
    colorscale=dict(sequential=[[0, LOW], [0.5, HIGH], [1, CRITICAL]]),
)
pio.templates["app_dark"] = _t
pio.templates.default = "plotly+app_dark"
# Auto-export every figure to plots/ on show (overwrites previous run)
import os as _os, re as _re
import plotly.basedatatypes as _base
_os.makedirs("plots", exist_ok=True)
_plot_n = [0]
_orig_show = _base.BaseFigure.show
def _fixed_show(self, *args, **kwargs):
    _plot_n[0] += 1
    raw = (self.layout.title.text or f"plot_{_plot_n[0]}").strip()
    slug = _re.sub(r"[^\w]+", "_", raw).strip("_")
    self.write_image(f"plots/{slug}.png")
    self.write_image(f"plots/{slug}.svg")
    return _orig_show(self, *args, **kwargs)
_base.BaseFigure.show = _fixed_show

## Geographic Clustering

In [18]:
d1 = ds1.groupby("SEARCH1_E").size().rename("total")
d2 = ds2_enriched.groupby("SEARCH1_E").size().rename("noticed")
geo = pd.concat([d1, d2], axis=1).fillna(0).reset_index()
geo["rate"] = geo["noticed"] / geo["total"]
geo = geo.sort_values("rate", ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="DS1 share", x=geo["SEARCH1_E"],
    y=geo["total"] / geo["total"].sum(),
    marker_color=ACCENT, opacity=0.8,
))
fig.add_trace(go.Bar(
    name="DS2 share", x=geo["SEARCH1_E"],
    y=geo["noticed"] / geo["noticed"].sum(),
    marker_color=CRITICAL, opacity=0.8,
))
fig.update_layout(
    barmode="group",
    title="District share: all buildings (DS1) vs noticed buildings (DS2)",
    xaxis_title="District", yaxis_title="Share of dataset",
    xaxis_tickangle=-40,
)
fig.show()

fig2 = px.bar(
    geo, x="SEARCH1_E", y="rate",
    title="Statutory notice rate by district (DS2 / DS1)",
    labels={"rate": "Notice rate", "SEARCH1_E": "District"},
    color="rate", color_continuous_scale=[[0, LOW], [1, CRITICAL]],
)
fig2.update_layout(xaxis_tickangle=-40)
fig2.show()

Inspection notices are heavily concentrated in the old urban Kowloon core. **Yau Tsim Mong** stands out dramatically with a 31.4% notice rate — nearly one in three buildings there has received a statutory inspection or repair order — despite holding only 8.3% of the total building stock. **Sham Shui Po** (6.6%) and **Central & Western** (6.0%) follow, both high-density districts with significant pre-1970s building stock. In contrast, New Territories districts — which account for a substantial share of total buildings — show rates well below 1%, with Kwun Tong and Wong Tai Sin recording zero notices in this dataset. This geographic skew reflects the combination of age and density: tightly packed older tenement blocks in Kowloon are both more likely to meet the MBIS age threshold and harder to maintain, creating a compounding enforcement pressure in the same postcodes.

### By Region

In [19]:
r1 = ds1.groupby("SEARCH2_E").size().rename("total")
r2 = ds2_enriched.groupby("SEARCH2_E").size().rename("noticed")
reg = pd.concat([r1, r2], axis=1).fillna(0).reset_index()
reg["rate"] = reg["noticed"] / reg["total"]
reg = reg.sort_values("rate", ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="DS1 share", x=reg["SEARCH2_E"],
    y=reg["total"] / reg["total"].sum(),
    marker_color=ACCENT, opacity=0.8,
))
fig.add_trace(go.Bar(
    name="DS2 share", x=reg["SEARCH2_E"],
    y=reg["noticed"] / reg["noticed"].sum(),
    marker_color=CRITICAL, opacity=0.8,
))
fig.update_layout(
    barmode="group",
    title="Region share: all buildings (DS1) vs noticed buildings (DS2)",
    xaxis_title="Region", yaxis_title="Share of dataset",
)
fig.show()

fig2 = px.bar(
    reg, x="SEARCH2_E", y="rate",
    title="Statutory notice rate by region (DS2 / DS1)",
    labels={"rate": "Notice rate", "SEARCH2_E": "Region"},
    color="rate", color_continuous_scale=[[0, LOW], [1, CRITICAL]],
)
fig2.show()

The regional picture amplifies the district-level finding. **Kowloon** has a 12.5% notice rate and accounts for **79% of all noticed buildings** while holding only 25% of the total stock — a 3× over-representation. **Hong Kong Island** is moderately elevated at 2.4%, contributing 16% of DS2. The **New Territories**, despite containing nearly half of all buildings (47%), produce just 4.9% of notices at a 0.4% rate. This 30-fold difference between Kowloon and the New Territories confirms that region alone is a strong prior for inspection risk, driven by the concentration of pre-1970s unrenewed walk-up blocks in the Kowloon peninsula.

## Building Type vs Inspection Count

In [20]:
t1 = ds1.groupby("NSEARCH4_E").size().rename("total")
t2 = ds2_enriched.groupby("NSEARCH4_E").size().rename("noticed")
types = pd.concat([t1, t2], axis=1).fillna(0).reset_index()
types["rate"] = types["noticed"] / types["total"]

fig = go.Figure()
fig.add_trace(go.Bar(
    name="DS1 share", x=types["NSEARCH4_E"],
    y=types["total"] / types["total"].sum(),
    marker_color=ACCENT, opacity=0.8,
))
fig.add_trace(go.Bar(
    name="DS2 share", x=types["NSEARCH4_E"],
    y=types["noticed"] / types["noticed"].sum(),
    marker_color=CRITICAL, opacity=0.8,
))
fig.update_layout(
    barmode="group",
    title="Building type share: all buildings (DS1) vs noticed buildings (DS2)",
    xaxis_title="Building type", yaxis_title="Share of dataset",
)
fig.show()

avg_notices = ds2_enriched.groupby("NSEARCH4_E")["NSEARCH01_EN"].mean().reset_index()
fig2 = px.bar(
    avg_notices, x="NSEARCH4_E", y="NSEARCH01_EN",
    title="Average number of notices issued by building type",
    labels={"NSEARCH01_EN": "Avg notices", "NSEARCH4_E": "Building type"},
    color="NSEARCH4_E", color_discrete_sequence=[ACCENT, CRITICAL],
)
fig2.update_layout(showlegend=False)
fig2.show()

Podiums are over-represented among noticed buildings relative to their share of the total stock. They make up 12.3% of DS1 but 18.3% of DS2, giving a notice rate of 6.0% compared to 3.8% for Towers. The disparity becomes sharper when looking at enforcement intensity: Podiums that have already received a notice average **8.4 notices each**, versus **5.9 for Towers**. This suggests Podiums are not only more likely to attract an initial notice but also more likely to accumulate repeat orders — consistent with their structural complexity (mixed-use plinths typically serve both retail and residential loads) making remediation harder to complete and sign off. For a risk model, building type should be treated as a multiplier on notice probability rather than an independent predictor: a Podium in Yau Tsim Mong aged over 50 years combines all three high-risk factors simultaneously.

## Annual Inspection Load Forecast

In [21]:
fig = px.histogram(
    ds1, x="age", nbins=50,
    title="Building age distribution (DS1)",
    labels={"age": "Building age (years)"},
    color_discrete_sequence=[ACCENT],
)
fig.add_vline(x=30, line_dash="dash", line_color=CRITICAL, annotation_text="30-year threshold")
fig.update_layout(bargap=0.05)
fig.show()

In [22]:
years = ds1["completion_date"].dt.year.dropna().astype(int)

year_range = range(years.min() + 30, years.max() + 31)
load = pd.DataFrame({
    "year": year_range,
    "load": [(years <= yr - 30).sum() / 10 for yr in year_range],
})

today = pd.Timestamp.today().year
fig = px.bar(
    load, x="year", y="load",
    color=load["year"].apply(lambda y: "past" if y < today else "forecast"),
    color_discrete_map={"past": ACCENT, "forecast": CRITICAL},
    title="Estimated annual inspection load (all buildings over 30 years old / 10)",
    labels={"year": "Year", "load": "Buildings to inspect per year", "color": ""},
)
fig.add_vline(x=today, line_dash="dash", line_color=TEXT1, annotation_text="today")
fig.update_layout(bargap=0)
fig.show()